In [ ]:
%%configure  
{ "vCores": {  "parameterName": "pipelinecore",  "defaultValue": 2 }}

In [ ]:
ws         = '1c52481c-0523-4a5a-bbde-fdc932bd77c2'          
lh         = 'ac303243-4441-4885-9e7d-f4f5e7af194c'  
results    = 'abfss://sqlengines@onelake.dfs.fabric.microsoft.com/benchmarks.Lakehouse/Tables/dbo/result'  
SF         =  2
engine     = 'duckdb_iceberg'      

In [ ]:
import json
with open('/tmp/tpch_params.json', 'w') as f:
    json.dump({'SF': SF, 'engine': engine, 'ws':ws,'lh':lh,'results':results}, f)

In [ ]:
if engine == 'duckdb_iceberg':
    !pip install -q duckdb --upgrade
elif engine == 'polars_iceberg':
    !pip install -q polars --upgrade
elif engine in ('lakesail_iceberg'):
    !pip install -q grpcio-status==1.48.2
    !pip install -q pysail
!pip install pyiceberg
notebookutils.session.restartPython()

In [ ]:
import pandas    as pd
from   datetime  import datetime
from   psutil    import *
from   pyiceberg.catalog import load_catalog
import time
import json
import os

In [ ]:
with open('/tmp/tpch_params.json', 'r') as f:
    params  = json.load(f)
SF          = params['SF']
engine      = params['engine']
ws          = params['ws']
lh          = params['lh']
results     = params['results']
schema      = f'CH{SF:04d}'

In [ ]:
token       =    notebookutils.credentials.getToken('storage')
Endpoint    =   "https://onelake.table.fabric.microsoft.com/iceberg"
warehouse   =   f'{ws}/{lh}'

In [ ]:
os.environ["RUST_LOG"] = "error"

# Generate Data

In [ ]:
import os, sys, time, shutil, threading, subprocess, tempfile, importlib.util
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

def generate_data(sf, *, ws=None, lh=None, schema=None, token=None,
                  endpoint=None, warehouse=None,
                  max_local_parts=4, upload_threads=6, upload_concurrency=8,
                  pyiceberg_workers=16, extra_gen_args=(), force=False,
                  verbose=True):
    """Generate TPC-H at scale factor `sf` straight into OneLake as Iceberg.

    tpchgen-cli writes each parquet once, the bytes upload as-is, and the table
    resolves columns via a default name mapping - no re-encode, no field IDs,
    no query engine. Generation and upload overlap; `max_local_parts` caps how
    many parts sit on local disk. Returns a stats dict, or skips outright if
    the data is already there.
    """
    ws        = ws        or globals()["ws"]
    lh        = lh        or globals()["lh"]
    schema    = schema    or globals()["schema"]
    token     = token     or globals()["token"]
    endpoint  = endpoint  or globals()["Endpoint"]
    warehouse = warehouse or globals()["warehouse"]

    os.environ.setdefault("PYICEBERG_MAX_WORKERS", str(pyiceberg_workers))

    import pyarrow as pa
    import pyarrow.parquet as pq
    from pyiceberg.catalog import load_catalog
    from pyiceberg.table.name_mapping import create_mapping_from_schema

    base_path = f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"
    rel_root  = f"Tables/{schema}"
    namespace = schema
    t_start   = time.perf_counter()

    def log(msg):
        if verbose:
            print(msg, flush=True)

    catalog = load_catalog("onelake", **{
        "uri": endpoint, "token": token, "warehouse": warehouse,
        "adls.account-name": "onelake",
        "adls.account-host": "onelake.blob.fabric.microsoft.com",
        "adls.token": token,
    })

    # ---- bail out before installing anything -------------------------------
    if not force and catalog.table_exists(f"{namespace}.supplier"):
        log("Data already exists - skipping generation")
        return {"sf": sf, "skipped": True, "catalog": catalog,
                "namespace": namespace, "tables": {}, "elapsed_s": 0.0}

    # ---- only now pull in what generation actually needs --------------------
    missing = []
    if shutil.which("tpchgen-cli") is None:
        missing.append("tpchgen-cli")
    if importlib.util.find_spec("azure.storage.filedatalake") is None:
        missing.append("azure-storage-file-datalake")
    if missing:
        t0 = time.perf_counter()
        log(f"Installing {', '.join(missing)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                       check=True)
        os.environ["PATH"] += os.pathsep + str(Path(sys.executable).parent)
        importlib.invalidate_caches()
        log(f"  installed in {time.perf_counter() - t0:.1f}s")

    from azure.core.credentials import AccessToken
    from azure.storage.filedatalake import DataLakeServiceClient

    def scaled(base, floor=1):
        return max(floor, int(base * (sf / 1000)))

    tables_parts = {
        "lineitem": scaled(200, 2), "orders": scaled(140, 2),
        "partsupp": scaled(80), "part": scaled(12), "customer": scaled(20),
        "nation": 1, "region": 1, "supplier": 1,
    }

    class StaticToken:
        def __init__(self, tok): self._tok = tok
        def get_token(self, *scopes, **kw):
            return AccessToken(self._tok, int(time.time()) + 3600)

    svc = DataLakeServiceClient(
        account_url="https://onelake.dfs.fabric.microsoft.com",
        credential=StaticToken(token))
    fs = svc.get_file_system_client(ws)

    slots  = threading.Semaphore(max_local_parts)
    lock   = threading.Lock()
    timers = {"gen_s": 0.0, "upload_s": 0.0, "bytes": 0}

    def make_all_optional(s):
        return pa.schema([pa.field(f.name, f.type, nullable=True,
                                   metadata=f.metadata) for f in s])

    def ensure_table(name, sample):
        tbl = catalog.create_table_if_not_exists(
            identifier=f"{namespace}.{name}",
            schema=make_all_optional(pq.read_schema(sample)),
            location=f"{base_path}/{rel_root}/{name}")
        if "schema.name-mapping.default" not in tbl.properties:
            with tbl.transaction() as tx:
                tx.set_properties(**{
                    "schema.name-mapping.default":
                        create_mapping_from_schema(tbl.schema()).model_dump_json()})
        return tbl

    def upload_part(jobs, part_dir, stats):
        t0 = time.perf_counter()
        try:
            for local, rel, nbytes in jobs:
                with open(local, "rb") as fh:
                    fs.get_file_client(rel).upload_data(
                        fh, overwrite=True, max_concurrency=upload_concurrency)
                local.unlink(missing_ok=True)
                with lock:
                    timers["bytes"] += nbytes
                    stats["bytes"]  += nbytes
        finally:
            dt = time.perf_counter() - t0
            with lock:
                timers["upload_s"] += dt
                stats["upload_s"]  += dt
            shutil.rmtree(part_dir, ignore_errors=True)
            slots.release()

    def build_table(name, total_parts, temp_root, pool):
        stats = {"parts": total_parts, "files": 0, "rows": 0, "bytes": 0,
                 "gen_s": 0.0, "upload_s": 0.0, "register_s": 0.0}
        t_tbl, tbl, futures, registered = time.perf_counter(), None, [], []
        log(f"\n--- {name} ({total_parts} parts) ---")

        for part_num in range(1, total_parts + 1):
            slots.acquire()
            part_dir = temp_root / f"{name}-{part_num}"
            part_dir.mkdir(parents=True, exist_ok=True)
            t0 = time.perf_counter()
            try:
                subprocess.run(["tpchgen-cli", "-s", str(sf), "--tables", name,
                                "--output-dir", str(part_dir),
                                "--parts", str(total_parts), "--part", str(part_num),
                                "--format", "parquet", *extra_gen_args], check=True)
                files = sorted(part_dir.rglob("*.parquet"))
                if not files:
                    raise RuntimeError(f"{name} part {part_num}: no parquet produced")
            except Exception:
                shutil.rmtree(part_dir, ignore_errors=True)
                slots.release()
                raise
            dt = time.perf_counter() - t0
            with lock:
                timers["gen_s"] += dt
                stats["gen_s"]  += dt

            if tbl is None:
                tbl = ensure_table(name, files[0])

            jobs = []
            for i, f in enumerate(files):
                fname = f"part-{part_num:05d}-{i:03d}-{f.name}"
                stats["rows"]  += pq.read_metadata(f).num_rows
                stats["files"] += 1
                jobs.append((f, f"{lh}/{rel_root}/{name}/{fname}", f.stat().st_size))
                registered.append(f"{base_path}/{rel_root}/{name}/{fname}")
            futures.append(pool.submit(upload_part, jobs, part_dir, stats))
            log(f"  part {part_num}/{total_parts} generated ({dt:.1f}s), queued")

        for fut in futures:
            fut.result()

        t0 = time.perf_counter()
        tbl.add_files(registered, check_duplicate_files=False)
        stats["register_s"] = time.perf_counter() - t0
        stats["wall_s"] = time.perf_counter() - t_tbl
        log(f"  {stats['files']} files, {stats['rows']:,} rows, "
            f"{stats['bytes']/2**30:.2f} GiB, add_files {stats['register_s']:.1f}s, "
            f"wall {stats['wall_s']:.1f}s")
        return stats

    log(f"Generating TPC-H SF={sf} -> {rel_root}")
    catalog.create_namespace_if_not_exists(namespace)
    results = {}
    with tempfile.TemporaryDirectory() as td, ThreadPoolExecutor(upload_threads) as pool:
        for name, parts in tables_parts.items():
            results[name] = build_table(name, parts, Path(td), pool)

    elapsed = time.perf_counter() - t_start
    if verbose:
        print(f"\n{'table':<10}{'files':>7}{'rows':>16}{'GiB':>9}{'gen s':>9}"
              f"{'up s':>9}{'wall s':>9}")
        for n, s in results.items():
            print(f"{n:<10}{s['files']:>7}{s['rows']:>16,}{s['bytes']/2**30:>9.2f}"
                  f"{s['gen_s']:>9.1f}{s['upload_s']:>9.1f}{s['wall_s']:>9.1f}")
        print(f"\ntotal {elapsed:.1f}s wall | gen {timers['gen_s']:.1f}s | "
              f"upload {timers['upload_s']:.1f}s | {timers['bytes']/2**30:.2f} GiB | "
              f"{timers['bytes']/2**20/elapsed:.1f} MiB/s effective")
    return {"sf": sf, "skipped": False, "catalog": catalog, "namespace": namespace,
            "tables": results, "elapsed_s": elapsed,
            "gen_s": timers["gen_s"], "upload_s": timers["upload_s"],
            "bytes": timers["bytes"]}

In [ ]:
generate_data(SF)

# Attach Catalog

In [ ]:
start = time.time()
exclude_list=[]
if engine == 'duckdb_iceberg':
     import duckdb
     conn = duckdb.connect()
     conn.sql(f"""              
                                ATTACH or replace '{warehouse}' AS onelake
                                                                (TYPE ICEBERG,
                                                                ENDPOINT '{Endpoint}',
                                                                TOKEN '{token}',
                                                                MAX_TABLE_STALENESS '10 minutes',
                                                                ACCESS_DELEGATION_MODE 'none',
                                                                DEFAULT_SCHEMA '{schema}');
                                USE onelake ;
            """) 

elif engine == 'lakesail_iceberg':
    from pysail.spark import SparkConnectServer
    from pyspark.sql import SparkSession
    os.environ['SAIL_OPTIMIZER__ENABLE_JOIN_REORDER'] = 'true'
    os.environ['SAIL_EXECUTION__COLLECT_STATISTICS'] = 'true'
    os.environ['SAIL_CATALOG__LIST'] = (
                                            f'[{{type="onelake", name="onelake", url="{warehouse}", '
                                            f'api="iceberg", bearer_token="{token}", '
                                            f'table_cache_type="session", table_cache_ttl_secs=300, '
                                            f'database_cache_type="session", database_cache_ttl_secs=300}}]'
                                        )
    server = SparkConnectServer()
    server.start()
    _, port = server.listening_address
    conn = SparkSession.builder.remote(f"sc://localhost:{port}").getOrCreate()
    conn.sql(f" use schema {schema}")
    
elif engine == 'polars_iceberg':
    import polars as pl
    onelake_catalog = load_catalog("onelake", **{
        "uri": Endpoint, "token": token, "warehouse": warehouse,
        "adls.account-name": "onelake",
        "adls.account-host": "onelake.blob.fabric.microsoft.com",
        "adls.token": token,
    })
    conn = pl
    for tbl in ['lineitem','customer','nation','orders','part','partsupp','region','supplier']:
        globals()[tbl] = conn.scan_iceberg( onelake_catalog.load_table(f"{schema}.{tbl}"),storage_options= {"bearer_token": '{token}'}) 
setup_time = time.time() - start

# SQL Tests

In [ ]:
sql = (f'''
SELECT
    --Query01
    l_returnflag,
    l_linestatus,
    SUM(l_quantity) AS sum_qty,
    SUM(l_extendedprice) AS sum_base_price,
    SUM(l_extendedprice * (1 - l_discount)) AS sum_disc_price,
    SUM(l_extendedprice * (1 - l_discount) * (1 + l_tax)) AS sum_charge,
    AVG(l_quantity) AS avg_qty,
    AVG(l_extendedprice) AS avg_price,
    AVG(l_discount) AS avg_disc,
    COUNT(*) AS count_order
FROM
    lineitem
WHERE
    l_shipdate <= CAST('1998-09-02' AS date)
GROUP BY
    l_returnflag,
    l_linestatus
ORDER BY
    l_returnflag,
    l_linestatus;



WITH cheapest_part AS (
    SELECT
        MIN(ps.ps_supplycost) AS cp_lowest,
        p.p_partkey AS cp_partkey
    FROM part p
    JOIN partsupp ps ON p.p_partkey = ps.ps_partkey
    JOIN supplier s ON s.s_suppkey = ps.ps_suppkey
    JOIN nation n ON s.s_nationkey = n.n_nationkey
    JOIN region r ON n.n_regionkey = r.r_regionkey
    WHERE r.r_name = 'EUROPE'
    GROUP BY p.p_partkey
)
SELECT
    s.s_acctbal,
    s.s_name,
    n.n_name,
    p.p_partkey,
    p.p_mfgr,
    s.s_address,
    s.s_phone,
    s.s_comment
FROM part p
JOIN partsupp ps ON p.p_partkey = ps.ps_partkey
JOIN supplier s ON s.s_suppkey = ps.ps_suppkey
JOIN nation n ON s.s_nationkey = n.n_nationkey
JOIN region r ON n.n_regionkey = r.r_regionkey
JOIN cheapest_part cp ON ps.ps_supplycost = cp.cp_lowest AND cp.cp_partkey = p.p_partkey
WHERE p.p_size = 15
  AND p.p_type LIKE '%BRASS'
  AND r.r_name = 'EUROPE'
ORDER BY s.s_acctbal DESC,
         n.n_name,
         s.s_name,
         p.p_partkey
LIMIT 10;



SELECT
    l.l_orderkey,
    SUM(l.l_extendedprice * (1 - l.l_discount)) AS revenue,
    o.o_orderdate,
    o.o_shippriority
FROM
    customer c
JOIN orders o ON c.c_custkey = o.o_custkey
JOIN lineitem l ON l.l_orderkey = o.o_orderkey
WHERE
    c.c_mktsegment = 'BUILDING'
    AND o.o_orderdate < CAST('1995-03-15' AS DATE)
    AND l.l_shipdate > CAST('1995-03-15' AS DATE)
GROUP BY
    l.l_orderkey,
    o.o_orderdate,
    o.o_shippriority
ORDER BY
    revenue DESC,
    o.o_orderdate
LIMIT 10;



select
--Query04
    o_orderpriority,
    count(*) as order_count
from
    orders
where
    o_orderdate >= cast('1993-07-01' as date)
    and o_orderdate < cast('1993-10-01' as date)
    and o_orderkey in (
        select
            l_orderkey
        from
            lineitem
        where
            l_commitdate < l_receiptdate
    )
group by
    o_orderpriority
order by
    o_orderpriority;




SELECT
    --Query05
    n_name,
    SUM(l_extendedprice * (1 - l_discount)) AS revenue
FROM lineitem
inner join (select * from orders where o_orderdate >= '1994-01-01' AND o_orderdate < '1995-01-01') as x
on l_orderkey = x.o_orderkey
left join supplier
on l_suppkey = s_suppkey
left join customer
on o_custkey = c_custkey and
c_nationkey = s_nationkey
left join nation
on s_nationkey = n_nationkey
inner join ( select * from region where r_name = 'ASIA') as xx
on n_regionkey = xx.r_regionkey

GROUP BY
    n_name
ORDER BY
    revenue DESC;


SELECT
    --Query06
    SUM(l_extendedprice * l_discount) AS revenue
FROM
    lineitem
WHERE
    l_shipdate >= CAST('1994-01-01' AS date)
    AND l_shipdate < CAST('1995-01-01' AS date)
    AND l_discount BETWEEN 0.05
    AND 0.07
    AND l_quantity < 24;







SELECT
    --Query07
    supp_nation,
    cust_nation,
    l_year,
    SUM(volume) AS revenue
FROM (
    SELECT
        n1.n_name AS supp_nation,
        n2.n_name AS cust_nation,
        EXTRACT(YEAR FROM l.l_shipdate) AS l_year,
        l.l_extendedprice * (1 - l.l_discount) AS volume
    FROM
        supplier s
    JOIN lineitem l ON s.s_suppkey = l.l_suppkey
    JOIN orders o ON o.o_orderkey = l.l_orderkey
    JOIN customer c ON c.c_custkey = o.o_custkey
    JOIN nation n1 ON s.s_nationkey = n1.n_nationkey
    JOIN nation n2 ON c.c_nationkey = n2.n_nationkey
    WHERE
        (n1.n_name = 'FRANCE' AND n2.n_name = 'GERMANY')
        OR (n1.n_name = 'GERMANY' AND n2.n_name = 'FRANCE')
        AND l.l_shipdate BETWEEN CAST('1995-01-01' AS DATE) AND CAST('1996-12-31' AS DATE)
) AS shipping
GROUP BY
    supp_nation,
    cust_nation,
    l_year
ORDER BY
    supp_nation,
    cust_nation,
    l_year;








SELECT
    --Query08
        EXTRACT( year  FROM  o_orderdate ) AS o_year,
        SUM(  CASE  WHEN n2.n_name = 'BRAZIL' THEN l_extendedprice * (1 - l_discount) ELSE 0  END ) / SUM(l_extendedprice * (1 - l_discount)) AS mkt_share
        FROM  lineitem
        inner join   (select o_custkey,o_orderdate, o_orderkey from  orders WHERE  o_orderdate BETWEEN CAST('1995-01-01' AS date) AND CAST('1996-12-31' AS date) ) xxx
        on l_orderkey = xxx.o_orderkey
        inner join  (select p_partkey from  part  where p_type = 'ECONOMY ANODIZED STEEL' ) z
        on  l_partkey = z.p_partkey
        left join    supplier
        on  l_suppkey = s_suppkey
        left join   customer
        on o_custkey = c_custkey
        left join   nation n1
        on c_nationkey = n1.n_nationkey
        left join   nation n2
        on s_nationkey = n2.n_nationkey
        inner join  (select * from region where r_name = 'AMERICA') cc
        on  n1.n_regionkey = cc.r_regionkey




GROUP BY
    o_year
ORDER BY
    o_year;










SELECT
    --Query09
    n_name AS nation,
    EXTRACT( year  FROM o_orderdate ) AS o_year,
    sum(l_extendedprice * (1 - l_discount) - ps_supplycost * l_quantity) AS sum_profit
        FROM lineitem
        inner join ( select p_partkey from part where  p_name LIKE '%green%') xx
        on  l_partkey = xx.p_partkey
        left join orders
        on  l_orderkey =o_orderkey
        left join  partsupp
        on  l_suppkey =ps_suppkey  AND  l_partkey = ps_partkey
        left join  supplier
        on    l_suppkey =s_suppkey
        left join nation
        on  n_nationkey = s_nationkey


GROUP BY
    n_name,
    o_year
ORDER BY
    n_name,
    o_year DESC;



SELECT
    --Query10
    c_custkey,
    c_name,
    SUM(l_extendedprice * (1 - l_discount)) AS revenue,
    c_acctbal,
    n_name,
    c_address,
    c_phone,
    c_comment
FROM  lineitem
inner join ( select * from orders where o_orderdate >= '1993-10-01' AND o_orderdate < '1994-01-01') as xx
on l_orderkey = xx.o_orderkey
left join customer
on xx.o_custkey = c_custkey
left join nation
on c_nationkey = n_nationkey
WHERE  l_returnflag = 'R'

GROUP BY
    c_custkey,
    c_name,
    c_acctbal,
    c_phone,
    n_name,
    c_address,
    c_comment
ORDER BY
    revenue DESC
LIMIT
    20;



WITH germany_value AS (
    SELECT
        SUM(ps.ps_supplycost * ps.ps_availqty) * (0.0001 / {SF}) AS threshold
    FROM
        partsupp ps
    JOIN supplier s ON ps.ps_suppkey = s.s_suppkey
    JOIN nation n ON s.s_nationkey = n.n_nationkey
    WHERE
        n.n_name = 'GERMANY'
),
partkey_values AS (
    SELECT
        ps.ps_partkey,
        SUM(ps.ps_supplycost * ps.ps_availqty) AS value
    FROM
        partsupp ps
    JOIN supplier s ON ps.ps_suppkey = s.s_suppkey
    JOIN nation n ON s.s_nationkey = n.n_nationkey
    WHERE
        n.n_name = 'GERMANY'
    GROUP BY
        ps.ps_partkey
)
SELECT pv.ps_partkey, pv.value
FROM partkey_values pv
CROSS JOIN germany_value gv
WHERE pv.value > gv.threshold
ORDER BY pv.value DESC;







SELECT
    --Query12
    l_shipmode,
    SUM(
        CASE
            WHEN o_orderpriority = '1-URGENT'
            OR o_orderpriority = '2-HIGH' THEN 1
            ELSE 0
        END
    ) AS high_line_count,
    SUM(
        CASE
            WHEN o_orderpriority <> '1-URGENT'
            AND o_orderpriority <> '2-HIGH' THEN 1
            ELSE 0
        END
    ) AS low_line_count
FROM lineitem
left join  orders
on o_orderkey = l_orderkey

WHERE  l_shipmode IN ('MAIL', 'SHIP')
       AND l_commitdate < l_receiptdate
       AND l_shipdate < l_commitdate
       AND l_receiptdate >=  '1994-01-01'  AND l_receiptdate < '1995-01-01'
GROUP BY
    l_shipmode
ORDER BY
    l_shipmode;








--Query13
SELECT
    c_count,
    COUNT(*) AS custdist
FROM
    (
        SELECT
            c_custkey,
            COUNT(o_orderkey) AS c_count
        FROM
            customer
            LEFT OUTER JOIN (
                SELECT o_custkey, o_orderkey
                FROM orders
                WHERE o_comment NOT LIKE '%special%requests%'
            ) AS filtered_orders ON c_custkey = o_custkey
        GROUP BY
            c_custkey
    ) AS c_orders
GROUP BY
    c_count
ORDER BY
    custdist DESC,
    c_count DESC;








SELECT
    --Query14
    100.00 * SUM(
        CASE
            WHEN p_type LIKE 'PROMO%' THEN l_extendedprice * (1 - l_discount)
            ELSE 0
        END
    ) / SUM(l_extendedprice * (1 - l_discount)) AS promo_revenue
FROM  lineitem
left join part
on l_partkey = p_partkey
WHERE l_shipdate >= cast('1995-09-01' as date) AND l_shipdate < cast('1995-10-01' as date);








--Query15
WITH revenue AS (
    SELECT
        l_suppkey AS supplier_no,
        SUM(l_extendedprice * (1 - l_discount)) AS total_revenue
    FROM
        lineitem
    WHERE
        l_shipdate >= CAST('1996-01-01' AS date)
        AND l_shipdate < CAST('1996-04-01' AS date)
    GROUP BY
        l_suppkey
)
SELECT
    s_suppkey,
    s_name,
    s_address,
    s_phone,
    total_revenue
FROM
    supplier
JOIN revenue ON s_suppkey = supplier_no
WHERE
    total_revenue IN (
        SELECT MAX(total_revenue)
        FROM revenue
    )
ORDER BY
    s_suppkey;







SELECT
    --Query16
    p.p_brand,
    p.p_type,
    p.p_size,
    COUNT(DISTINCT ps.ps_suppkey) AS supplier_cnt
FROM
    partsupp ps
JOIN part p ON p.p_partkey = ps.ps_partkey
WHERE
    p.p_brand <> 'Brand#45'
    AND p.p_type NOT LIKE 'MEDIUM POLISHED%'
    AND p.p_size IN (49, 14, 23, 45, 19, 3, 36, 9)
    AND ps.ps_suppkey NOT IN (
        SELECT
            s.s_suppkey
        FROM
            supplier s
        WHERE
            s.s_comment LIKE '%Customer%Complaints%'
    )
GROUP BY
    p.p_brand,
    p.p_type,
    p.p_size
ORDER BY
    supplier_cnt DESC,
    p.p_brand,
    p.p_type,
    p.p_size;









WITH part_avg AS (
    -- Query17
    SELECT
        (0.2 * AVG(l.l_quantity)) AS limit_qty,
        l.l_partkey AS lpk
    FROM lineitem l
    GROUP BY l.l_partkey
)
SELECT
    SUM(l.l_extendedprice) / 7.0 AS avg_yearly
FROM
    lineitem l
JOIN part p ON p.p_partkey = l.l_partkey
JOIN part_avg pa ON p.p_partkey = pa.lpk
WHERE
    p.p_brand = 'Brand#23'
    AND p.p_container = 'MED BOX'
    AND l.l_quantity < pa.limit_qty;







SELECT
    --Query18
    c.c_name,
    c.c_custkey,
    o.o_orderkey,
    o.o_orderdate,
    o.o_totalprice,
    SUM(l.l_quantity)
FROM
    customer c
JOIN orders o ON c.c_custkey = o.o_custkey
JOIN lineitem l ON o.o_orderkey = l.l_orderkey
WHERE
    o.o_orderkey IN (
        SELECT
            l_orderkey
        FROM
            lineitem
        GROUP BY
            l_orderkey
        HAVING
            SUM(l_quantity) > 300
    )
GROUP BY
    c.c_name,
    c.c_custkey,
    o.o_orderkey,
    o.o_orderdate,
    o.o_totalprice
ORDER BY
    o.o_totalprice DESC,
    o.o_orderdate
LIMIT
    100;






select
--Query19
sum(l_extendedprice* (1 - l_discount)) as revenue

from lineitem
join  part
ON p_partkey = l_partkey
where (
        p_brand = 'Brand#12'
        and p_container in ('SM CASE', 'SM BOX', 'SM PACK', 'SM PKG')
        and l_quantity >= 1 and l_quantity <= 1 + 10
        and p_size between 1 and 5
        and l_shipmode in ('AIR', 'AIR REG')
        and l_shipinstruct = 'DELIVER IN PERSON'
    ) or ( p_partkey = l_partkey
        and p_brand = 'Brand#23'
        and p_container in ('MED BAG', 'MED BOX', 'MED PKG', 'MED PACK')
        and l_quantity >= 10 and l_quantity <= 10 + 10
        and p_size between 1 and 10
        and l_shipmode in ('AIR', 'AIR REG')
        and l_shipinstruct = 'DELIVER IN PERSON'
    ) or ( p_partkey = l_partkey
        and p_brand = 'Brand#34'
        and p_container in ('LG CASE', 'LG BOX', 'LG PACK', 'LG PKG')
        and l_quantity >= 20 and l_quantity <= 20 + 10
        and p_size between 1 and 15
        and l_shipmode in ('AIR', 'AIR REG')
        and l_shipinstruct = 'DELIVER IN PERSON'
    );





--Query20
WITH availability_part_supp AS (
    SELECT 
        0.5 * SUM(l_quantity) AS ps_halfqty, 
        l_partkey AS pkey, 
        l_suppkey AS skey
    FROM lineitem
    WHERE l_shipdate >= CAST('1994-01-01' AS date)
      AND l_shipdate < CAST('1995-01-01' AS date)
    GROUP BY l_partkey, l_suppkey
)
SELECT s_name, s_address
FROM supplier
JOIN nation ON s_nationkey = n_nationkey
WHERE s_suppkey IN (
    SELECT ps_suppkey
    FROM partsupp
    JOIN availability_part_supp ON ps_partkey = pkey AND ps_suppkey = skey
    WHERE ps_partkey IN (
        SELECT p_partkey
        FROM part
        WHERE p_name LIKE 'forest%'
    )
    AND ps_availqty > ps_halfqty
)
AND n_name = 'CANADA'
ORDER BY s_name;




SELECT
    --Query21
    s.s_name,
    COUNT(*) AS numwait
FROM
    supplier s
JOIN lineitem l1 ON s.s_suppkey = l1.l_suppkey
JOIN orders o ON o.o_orderkey = l1.l_orderkey
JOIN nation n ON s.s_nationkey = n.n_nationkey
WHERE
    o.o_orderstatus = 'F'
    AND l1.l_receiptdate > l1.l_commitdate
    AND l1.l_orderkey IN (
        SELECT l_orderkey
        FROM lineitem
        GROUP BY l_orderkey
        HAVING COUNT(l_suppkey) > 1
    )
    AND l1.l_orderkey NOT IN (
        SELECT l_orderkey
        FROM lineitem
        WHERE l_receiptdate > l_commitdate
        GROUP BY l_orderkey
        HAVING COUNT(l_suppkey) > 1
    )
    AND n.n_name = 'SAUDI ARABIA'
GROUP BY s.s_name
ORDER BY numwait DESC, s.s_name
LIMIT 100;





--Query22
WITH avg_acctbal AS (
    SELECT AVG(c_acctbal) AS avg_bal
    FROM customer
    WHERE c_acctbal > 0
      AND SUBSTRING(c_phone FROM 1 FOR 2) IN ('13', '31', '23', '29', '30', '18', '17')
),
customers_with_orders AS (
    SELECT DISTINCT o_custkey
    FROM orders
)
SELECT
    cntrycode,
    COUNT(*) AS numcust,
    SUM(c_acctbal) AS totacctbal
FROM (
    SELECT
        SUBSTRING(c_phone FROM 1 FOR 2) AS cntrycode,
        c_acctbal
    FROM customer
    CROSS JOIN avg_acctbal
    WHERE SUBSTRING(c_phone FROM 1 FOR 2) IN ('13', '31', '23', '29', '30', '18', '17')
      AND c_acctbal > avg_bal
      AND c_custkey NOT IN (
          SELECT o_custkey FROM customers_with_orders
      )
) AS custsale
GROUP BY cntrycode
ORDER BY cntrycode;

''')
     


In [ ]:
def execute_query(conn, sql_script, engine, exclude_list=[]):
    results = []
    engine_lower = engine.lower()
    for index, value in enumerate(sql_script.split(";"), start=1):
        if index not in exclude_list and len(value.strip()) > 0:
            start = time.time()
            print('query' + str(index))
            conn.sql(value).show()
            duration = time.time() - start
            print(duration)
            results.append({'dur': duration, 'query': index})
    return pd.DataFrame(results) if results else pd.DataFrame(columns=['dur', 'query'])

In [ ]:
%%time
time_cold = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
df = execute_query(conn, sql, engine,[])

In [ ]:
%%time
time_warm = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
df1= execute_query(conn, sql, engine,[])

In [ ]:
from   deltalake import write_deltalake, DeltaTable
df.at[0, 'dur']  = df.at[0, 'dur'] + setup_time
df['Engine']     = engine
df['time']       = time_cold
df['sf']         = SF
df['cpu']        = cpu_count()
df['test']       = 'tpch'
df['type']       = 'cold'
df1['Engine']    = engine
df1['time']      = time_warm
df1['sf']        = SF
df1['cpu']       = cpu_count()
df1['test']      = 'tpch'
df1['type']      = 'warm'
write_deltalake(results, df, mode="append")
write_deltalake(results, df1,mode="append")
dt = DeltaTable(results)
dt.optimize.compact()
dt.vacuum(retention_hours=0,dry_run=False,  enforce_retention_duration=False)

# Results

In [ ]:
import duckdb
con = duckdb.connect()
raw = con.sql(f""" 
                    select Engine, time, query, sf, cpu as core, dur,type
                    from delta_scan('{results}')
                    where test = 'tpch'
            """)

In [ ]:
con.sql(f"""            
                        CREATE OR REPLACE TEMP TABLE cc AS
                        SELECT concat(Engine,'_', type) as Engine, query, AVG(dur) as dur
                        FROM raw
                        WHERE sf = {SF} AND time > '2026-05-01'
                        and core = {cpu_count()}
                        and Engine in ('duckdb_iceberg','lakesail_iceberg','polars_iceberg') 
                        and type in('cold')
                        GROUP BY ALL 
        """)
s = con.sql("""  PIVOT (select * from cc) 
    ON Engine IN ( SELECT Engine FROM cc GROUP BY Engine ORDER BY SUM(dur) DESC ) USING SUM(dur) """).df()
ax = s.plot.bar(
    rot=45,  x='query',
    title=f"Fabric Single node {cpu_count()} cores, TPCH like {SF} using the same 22 SQL Queries, using Onelake Iceberg Rest Catalog",
    ylabel='Duration per Query (Lower is better)', figsize=(18,8)
)
ax.set_ylim(bottom=0)
ax.grid(axis='y', linestyle='--', alpha=0.5)

In [ ]:
import seaborn as sns
from matplotlib.ticker import MultipleLocator

con.sql(f"""
    CREATE OR REPLACE TEMP TABLE cc AS
    SELECT Engine, type, query, AVG(dur) AS dur,
           date_trunc('day', CAST(time AS TIMESTAMP)) AS day
    FROM raw WHERE sf = {SF} and core = {cpu_count()} and type in('warm','cold') and Engine in ('duckdb_iceberg','lakesail_iceberg','polars_iceberg') 
    GROUP BY ALL
""")

df = con.sql("""
    SELECT Engine, type, day, SUM(dur) AS dur
    FROM cc GROUP BY ALL
    ORDER BY Engine
""").df()

ax = sns.lineplot(
    data=df, x='day', y='dur',
    hue='Engine',      # colour = engine
    style='type',      # solid vs dashed + marker shape = warm / cold
    markers=True, dashes=True, marker='o',
)
ax.set_title(f"Fabric Single node {cpu_count()} cores, TPCH like {SF},  OneLake Iceberg Catalog")
ax.set_ylabel('Total Duration (Lower is better)')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.figure.set_size_inches(18, 8)

In [ ]:
con.sql(f"""            
                        CREATE OR REPLACE TEMP TABLE cc AS
                        SELECT concat(Engine,'_', type) as Engine, query, AVG(dur) as dur ,core
                        FROM raw
                        where  sf = {SF} AND time > '2026-06-31' and core = {cpu_count()}
                        and type in('warm')
                        and Engine in ('duckdb_iceberg','lakesail_iceberg','polars_iceberg') 
                        GROUP BY ALL 
        """)
s = con.sql("""  PIVOT (select *exclude(query) from cc) 
    ON Engine IN ( SELECT Engine FROM cc  GROUP BY Engine ORDER BY SUM(dur) DESC ) USING SUM(dur) """).df()
ax = s.plot.bar(
    rot=0,
    x='core',
    title=f"Fabric Single node TPCH like {SF}  using Onelake Iceberg Catalog ",
    ylabel='Total Duration (Lower is better)', figsize=(18,8)
)
ax.set_ylim(bottom=0)
ax.grid(axis='y', linestyle='--', alpha=0.5)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=3)

In [ ]:
con.sql(f"""
    CREATE OR REPLACE TEMP TABLE cc AS
    SELECT split_part(Engine, '_', 1) as Engine, query, AVG(dur) AS dur,
           date_trunc('day', CAST(time AS TIMESTAMP)) AS day
    FROM raw
    WHERE sf = {SF} and core = {cpu_count()} 
    and Engine in ('duckdb_iceberg','lakesail_iceberg','polars_iceberg','duckdb','lakesail')
    GROUP BY ALL
""")
# total over the 22 queries, per engine/type/day  (matches your old SUM-in-pivot)
df = con.sql("""
    SELECT Engine,  day, SUM(dur) AS dur
    FROM cc GROUP BY ALL
    ORDER BY Engine
""").df()

ax = sns.lineplot(
    data=df, x='day',
    y='dur',
    hue='Engine',
    markers=True, dashes=True,
)
ax.set_title(f"Fabric Single node {cpu_count()} cores, TPCH like {SF}, average two runs (cold + warm)  OneLake Iceberg Catalog")
ax.set_ylabel('Total Duration (Lower is better)')
ax.set_ylim(bottom=0)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.figure.set_size_inches(18, 8)

In [ ]:
con.sql(f"""            
                        CREATE OR REPLACE TEMP TABLE cc AS
                        SELECT concat(Engine,'_', type) as Engine, query, AVG(dur) as dur ,core
                        FROM raw
                        where  sf = {SF} AND time > '2026-06-31' 
                        and type in('warm')
                        and Engine in ('duckdb_iceberg','lakesail_iceberg','polars_iceberg') 
                        GROUP BY ALL 
        """)
s = con.sql("""  PIVOT (select *exclude(query) from cc) 
    ON Engine IN ( SELECT Engine FROM cc  GROUP BY Engine ORDER BY SUM(dur) DESC ) USING SUM(dur) """).df()
ax = s.plot.bar(
    rot=0,
    x='core',
    title=f"Fabric Single node TPCH like {SF}  using Onelake Iceberg Catalog ",
    ylabel='Total Duration (Lower is better)', figsize=(18,8)
)
ax.set_ylim(bottom=0)
ax.grid(axis='y', linestyle='--', alpha=0.5)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', padding=3)